In [ ]:
!pip install -r requirements.txt

In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
from config import LLMKGKANConfig
from data import collate_fn
from model import LLMKGKAN
from torch.utils.data import DataLoader
import torch

from domains import SOURCE_DOMAINS, LOW_RESOURCE_TARGETS, ZERO_SHOT_TARGETS
from utils import build_datasets, set_seed
from metrics import compute_macro_f1
from kg_utils import ConceptNet

from transformers import get_linear_schedule_with_warmup
from difflib import SequenceMatcher

from torch.cuda.amp import autocast, GradScaler

In [ ]:
torch.set_default_dtype(torch.bfloat16)

In [ ]:
USE_FINETUNED = False   # 🔥 CHANGE THIS (True / False)

cfg = LLMKGKANConfig()
set_seed(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading KG...")
kg = ConceptNet("conceptnet-assertions-5.7.0.csv")

cfg.num_entities = len(kg.ent2id)
cfg.num_kg_relations = len(kg.rel2id)

print("KG loaded:", cfg.num_entities, "entities")

In [ ]:
def split_dataset(dataset, val_ratio=0.1):
    n = len(dataset)
    idx = list(range(n))
    split = int(n * (1 - val_ratio))

    return torch.utils.data.Subset(dataset, idx[:split]), torch.utils.data.Subset(dataset, idx[split:])

In [ ]:
def evaluate(model, loader):
    model.eval()
    scores = []

    device = next(model.parameters()).device

    with torch.no_grad():
        for batch in loader:

            # move only tensors to device
            batch = {
                k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in batch.items()
            }

            # AMP only for forward
            with autocast(dtype=torch.float16):
                out = model(batch)
                logits = out["logits"]

            labels = batch["labels"]

            f1 = compute_macro_f1(logits, labels)
            scores.append(f1)

    return sum(scores) / max(len(scores), 1)

In [ ]:
def train_for_domain(src, tgt, k_shot=None, seed=42):
    set_seed(seed)

    # ---- DATA ----
    src_data, tgt_data = build_datasets(cfg, kg, src, tgt, k_shot)

    tgt_train, tgt_val = split_dataset(tgt_data)

    src_loader = DataLoader(src_data, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn)
    tgt_loader = DataLoader(tgt_train, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn)

    val_loader = DataLoader(tgt_val, batch_size=cfg.batch_size, collate_fn=collate_fn)
    test_loader = DataLoader(tgt_data, batch_size=cfg.batch_size, collate_fn=collate_fn)

    total_steps = len(src_loader) * cfg.epochs

    # ---- MODEL ----
    model = LLMKGKAN(cfg).to(device)

    if USE_FINETUNED == False:
        # 🔥 ZERO-SHOT (NO TRAINING)
        model.eval()
        scores = []

        with torch.no_grad():
            for batch in test_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                out = model(batch)

                logits = out["logits"]
                labels = batch["labels"]

                f1 = compute_macro_f1(logits, labels)
                scores.append(f1)

        return sum(scores) / max(len(scores), 1)

    # ---- TRAINING ----
    scaler = GradScaler(enabled=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    best_f1 = 0
    patience = 3
    counter = 0

    for epoch in range(cfg.epochs):
        model.train()

        from itertools import cycle

        tgt_iter = cycle(tgt_loader)

        for src_batch in src_loader:
            tgt_batch = next(tgt_iter)

            src_batch = {k: v.to(device) for k, v in src_batch.items()}
            tgt_batch = {k: v.to(device) for k, v in tgt_batch.items()}

            # ---- MERGE SOURCE + TARGET ----
            batch = {}
            for k in src_batch:
                batch[k] = torch.cat([src_batch[k], tgt_batch[k]], dim=0)

            B = src_batch["input_ids"].size(0)

            batch["domain_ids"] = torch.cat([
                torch.zeros(B, dtype=torch.long),
                torch.ones(B, dtype=torch.long)
            ]).to(device)

            optimizer.zero_grad()

            with autocast(dtype=torch.float16):
                out = model(batch)
                loss = out["loss"]

            scaler.scale(loss).backward()

            # unscale before clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()

            scheduler.step()
        
        val_f1 = evaluate(model, val_loader)

        if val_f1 > best_f1:
            best_f1 = val_f1
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            print("Early stopping")
            break

    model_path = f"models/llmkgkan_{src}_to_{tgt}.pt"

    torch.save({
        "model_state_dict": model.state_dict(),
        "config": cfg.__dict__,
    }, model_path)

    print(f"Saved model to {model_path}")

    # ---- EVALUATION ----
    model.eval()
    scores = []

    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(batch)

            logits = out["logits"]
            labels = batch["labels"]

            f1 = compute_macro_f1(logits, labels)
            scores.append(f1)

    return sum(scores) / max(len(scores), 1)

In [ ]:
pairwise_results = {}

for tgt in LOW_RESOURCE_TARGETS:
    for src in SOURCE_DOMAINS:
        scores = []

        for seed in [1, 2, 3]:
            f1 = train_for_domain(src, tgt, k_shot=8, seed=seed)
            scores.append(f1)

        pairwise_results[f"{src}->{tgt}"] = sum(scores) / len(scores)

print(pairwise_results)

In [ ]:
USE_FINETUNED = False   # 🔥 IMPORTANT

few_shot_avg = {}

for tgt in LOW_RESOURCE_TARGETS:
    vals = [pairwise_results[f"{src}->{tgt}"] for src in SOURCE_DOMAINS]
    few_shot_avg[tgt] = sum(vals) / len(vals)

print("Few-shot avg:", few_shot_avg)

In [ ]:
USE_FINETUNED = False   # 🔥 IMPORTANT

zero_results = {}

for tgt in ZERO_SHOT_TARGETS:
    scores = []

    for src in SOURCE_DOMAINS:
        f1 = train_for_domain(src, tgt, k_shot=None)
        scores.append(f1)

    zero_results[tgt] = sum(scores) / len(scores)

print("Zero-shot:", zero_results)

In [ ]:
USE_FINETUNED = True

k_values = [2, 4, 8, 16]
k_results = {}

for k in k_values:
    scores = []

    for tgt in LOW_RESOURCE_TARGETS:
        for src in SOURCE_DOMAINS:
            f1 = train_for_domain(src, tgt, k_shot=k)
            scores.append(f1)

    k_results[k] = sum(scores) / len(scores)

print("K-shot:", k_results)